In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
HybridScore = object


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- readnext_hybrid_score_compare_scores_migration ---

# --- readnext_hybrid_score_to_frame_migration ---

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_readnext_hybrid_score_compare_scores_migration():
    def compare_hybrid_scores(*hybrid_scores: HybridScore) -> pd.DataFrame:
        """
        Stacks the hybrid recommender scores for multiple query documents vertically in a
        DataFrame.
        """
        return pd.concat([hybrid_score.to_frame() for hybrid_score in hybrid_scores], ignore_index=True)
    return compare_hybrid_scores

def before_readnext_hybrid_score_to_frame_migration():
    def to_frame(self) -> pd.DataFrame:
        """Collect all scores in a DataFrame."""
        return pd.DataFrame(
            {
                "Language Model": self.language_model_name,
                "Citation -> Language Candidates": round(
                    self.citation_to_language_candidates, ndigits=3
                ),
                "Citation -> Language Final": round(self.citation_to_language, ndigits=3),
                "Language -> Citation Candidates": round(
                    self.language_to_citation_candidates, ndigits=3
                ),
                "Language -> Citation Final": round(self.language_to_citation, ndigits=3),
            },
            index=[0],
        )
    return to_frame

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_readnext_hybrid_score_compare_scores_migration():
    def compare_hybrid_scores(*hybrid_scores: HybridScore) -> pl.DataFrame:
        """
        Stacks the hybrid recommender scores for multiple query documents vertically in a
        DataFrame.
        """
        return pl.concat([hybrid_score.to_frame() for hybrid_score in hybrid_scores], how="vertical")
    return compare_hybrid_scores

def gen_readnext_hybrid_score_to_frame_migration():
    def to_frame(self) -> pl.DataFrame:
        """Collect all scores in a DataFrame."""
        return pl.DataFrame(
            {
                "Language Model": [self.language_model_name],
                "Citation -> Language Candidates": [round(
                    self.citation_to_language_candidates, ndigits=3
                )],
                "Citation -> Language Final": [round(self.citation_to_language, ndigits=3)],
                "Language -> Citation Candidates": [round(
                    self.language_to_citation_candidates, ndigits=3
                )],
                "Language -> Citation Final": [round(self.language_to_citation, ndigits=3)],
            }
        )
    return to_frame

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: readnext_hybrid_score_to_frame_migration ===

# L1 smoke – generated
try:
    _r = gen_readnext_hybrid_score_to_frame_migration()
    print("✅ L1 smoke gen_readnext_hybrid_score_to_frame_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_readnext_hybrid_score_to_frame_migration: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_readnext_hybrid_score_to_frame_migration()
    print("✅ L1 smoke before_readnext_hybrid_score_to_frame_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_readnext_hybrid_score_to_frame_migration: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – execute returned to_frame method.
try:
    _before_fn = before_readnext_hybrid_score_to_frame_migration()
    _gen_fn = gen_readnext_hybrid_score_to_frame_migration()
    _score = SimpleNamespace(
        language_model_name="model",
        citation_to_language_candidates=0.12345,
        citation_to_language=0.98765,
        language_to_citation_candidates=1.23456,
        language_to_citation=9.87654,
    )
    _before = _before_fn(_score)
    _gen = _gen_fn(_score)
    compare(_before, _gen, "readnext_hybrid_score_to_frame_migration")
except Exception as _e:
    print(f"❌ L2 equivalence readnext_hybrid_score_to_frame_migration: setup error — {type(_e).__name__}: {_e}")

# L3 edge – zero values preserve schema and rounding.
try:
    _before_fn = before_readnext_hybrid_score_to_frame_migration()
    _gen_fn = gen_readnext_hybrid_score_to_frame_migration()
    _score = SimpleNamespace(
        language_model_name="edge-model",
        citation_to_language_candidates=0.0,
        citation_to_language=0.0,
        language_to_citation_candidates=0.0,
        language_to_citation=0.0,
    )
    _before_edge = _before_fn(_score)
    _gen_edge = _gen_fn(_score)
    compare(_before_edge, _gen_edge, "L3 edge readnext_hybrid_score_to_frame_migration")
except Exception as _e:
    print(f"❌ L3 edge readnext_hybrid_score_to_frame_migration: {type(_e).__name__}: {_e}")
